# COMP9085 Week 9 Laboratory
## Clusters, and whether to believe them

**Two hours.** In the lecture you saw that every clustering algorithm returns an answer, and that the hard part is
deciding whether to believe it. Today you cluster real biological data, choose the number of clusters defensibly,
and then try seriously to break your own result: with resampling, with a different algorithm, with a manufactured
batch effect, and with a second kind of biological data.

**Data.**
* **WDBC** (UCI Breast Cancer Wisconsin, Diagnostic): 569 tumours, 30 measurements of cell nuclei from a fine-needle
  aspirate image (Weeks 2, 3 and 7). It ships with scikit-learn, so no download is needed. The diagnosis (malignant or
  benign) is **not used for clustering**. We use it only afterwards, to check what was found. In a real discovery study
  you would not have it.
* **The Week 7 proteins**: 2,000 human Swiss-Prot proteins (UniProt release 2026_03, one per UniRef50 cluster) with
  their 23 classical descriptors and their ESM-2 (8M) embeddings. Files in `data/` (provenance in `data/README.md`).

**Learning objectives.** By the end you should be able to:
1. carry out k-means and a silhouette calculation by hand, and check them against scikit-learn;
2. justify a choice of k with the elbow, silhouette and bootstrap stability together;
3. tune DBSCAN with a k-distance plot and explain its behaviour in high dimension;
4. compare linkage criteria and recognise when a high silhouette is meaningless;
5. show that a technical batch effect can replace the biology, and why correction fails under confounding;
6. explain why clusters in an embedding space need not follow a property the embedding encodes;
7. write a defensible, honest statement of what a clustering result shows.

**How to work.** Core tasks 1–9 fit in two hours (times are a guide). Extensions A–C are optional. Code cells marked
`# your code here` are yours; cells marked `# provided` are complete. Every task ends with a question to answer in the
markdown cell below it: **the written answers matter more than the code.** The matching website exercises are on the
Week 9 page: https://hafli.github.io/comp9085-ml-biology/weeks/week09.html

**Deliverable.** This notebook, run top to bottom (**Kernel → Restart & Run All**), with every
question answered in the markdown cells.

**Runs in** Google Colab or any Python 3.10+ environment with numpy, pandas, matplotlib, scipy and scikit-learn
(≥ 1.3). No GPU and no deep-learning package are needed. The whole notebook runs in about a minute.

In [ ]:
# provided: setup
import time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
import sklearn
warnings.filterwarnings("ignore")
SEED = 42
print("scikit-learn", sklearn.__version__)

# WDBC: X raw measurements, y diagnosis (1 = malignant) kept aside for checking only
wdbc = load_breast_cancer()
X_raw = wdbc.data
names = list(wdbc.feature_names)
y = (wdbc.target == 0).astype(int)
Z = StandardScaler().fit_transform(X_raw)
print(X_raw.shape, "tumours x features;", y.sum(), "malignant,", (1 - y).sum(), "benign")

# The Week 7 proteins: local data/ folder, or the course website (for Colab)
URL = "https://hafli.github.io/comp9085-ml-biology/labs/week09/data/"
DATA = next((p for p in [Path("data"), Path("../data")] if (p / "proteins_classical_features.csv").exists()), None)
if DATA is None:
    import urllib.request
    DATA = Path("data"); DATA.mkdir(exist_ok=True)
    for f in ["proteins_classical_features.csv", "esm2_t6_8M_embeddings.csv.gz"]:
        urllib.request.urlretrieve(URL + f, DATA / f)
prot = pd.read_csv(DATA / "proteins_classical_features.csv")
emb = pd.read_csv(DATA / "esm2_t6_8M_embeddings.csv.gz").set_index("accession").loc[prot.accession]
print(len(prot), "proteins;", emb.shape[1], "embedding dimensions")

## Task 1. k-means and the silhouette by hand (10 min)

Six samples with the expression of one gene: `1, 2, 3, 8, 9, 12`. Start k-means with k = 2 from the (poor) centroids
1 and 2, exactly as in the lecture.

**1a.** Write the two steps of Lloyd's algorithm yourself in numpy: assign each value to its nearest centroid, then move
each centroid to the mean of its members. Print the assignment, the centroids and the SSE after every iteration, and
stop when nothing changes.

**1b.** Check your answer with `KMeans(n_clusters=2, init=np.array([[1.], [2.]]), n_init=1)`.

**1c.** Compute the silhouette of the value 3 by hand (in code, without `silhouette_samples`), then check it and the mean
silhouette with scikit-learn.

In [ ]:
x = np.array([1., 2., 3., 8., 9., 12.])
c = np.array([1., 2.])
# your code here


**Question 1.** Which value has the lowest silhouette, and why? Would a start at 1 and 12 have reached the same answer?

*Your answer here.*

## Task 2. Does scaling change the clusters? (15 min)

**2a.** Run `KMeans(n_clusters=2, n_init=10, random_state=SEED)` on the **raw** WDBC measurements and on the
**standardised** ones (`Z`). For each, report the silhouette (computed in the same space the algorithm used) and the
adjusted Rand index (ARI) with the diagnosis.

**2b.** Which two features hold most of the raw variance, and what share?

In [ ]:
# your code here


**Question 2.** The raw clustering has the *higher* silhouette. Is it the better clustering? Explain in two or three sentences.

*Your answer here.*

## Task 3. Choosing k: elbow and silhouette (15 min)

**3a.** For k = 1 to 10 on `Z`, record the WCSS (`inertia_`), and for k ≥ 2 the mean silhouette and the cluster sizes.
Plot WCSS and silhouette against k.

**3b.** Draw silhouette plots for k = 2 and k = 4 with the helper below. How many tumours have s(i) < 0?

**3c.** Run k-means at k = 4 from 50 different single starts (`n_init=1, init="random", random_state=s`). How many
distinct final WCSS values do you get?

**Decide your k now, before Task 4 shows the diagnosis.**

In [ ]:
# provided: silhouette plot helper
def silhouette_plot(ax, X, labels, title=""):
    sv = silhouette_samples(X, labels)
    y0 = 0
    for c in np.unique(labels):
        v = np.sort(sv[labels == c])
        ax.fill_betweenx(np.arange(y0, y0 + len(v)), 0, v, alpha=0.8)
        ax.text(-0.1, y0 + len(v) / 2, f"n={len(v)}", fontsize=8)
        y0 += len(v) + 10
    ax.axvline(sv.mean(), color="red", ls="--")
    ax.set_title(f"{title} mean {sv.mean():.3f}; {(sv < 0).sum()} below 0"); ax.set_yticks([])
    return sv

In [ ]:
# your code here


**Question 3.** Which k do you choose, and why? What does the size of the best silhouette tell you?

*Your answer here.*

## Task 4. Stability, then the label (10 min)

**4a.** Write `stability(Z, k, B=50)`: for each of B bootstrap resamples (rows drawn with replacement), fit k-means
(`n_init=10, random_state=b`) on the resample, assign **all** 569 tumours with `.predict`, and compute the ARI with the
full-data clustering. Return the B values. Use one generator, `rng = np.random.default_rng(SEED)`, created once before
looping over k = 2 to 6 in order (so your numbers match the lecture).

**4b.** Now reveal the label: the ARI between the k-means clusters and the diagnosis for k = 2 to 6, and the
2 × 2 table at k = 2.

In [ ]:
rng = np.random.default_rng(SEED)
# your code here


**Question 4.** Is a stable clustering necessarily a biological one? Give an example where it is not.

*Your answer here.*

## Task 5. DBSCAN and the k-distance plot (15 min)

**5a.** With `min_samples = 5`, compute each tumour's distance to its 4th nearest neighbour (`NearestNeighbors(n_neighbors=5)`;
the first neighbour is the point itself) in the 30-D space `Z` and on the first two principal components `Z2`
(provided). Plot both sorted curves.

**5b.** Run DBSCAN for eps in `[1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]` on `Z` and in `[0.3, 0.5, 0.75, 1.0, 1.25, 1.5]` on
`Z2`. Tabulate the number of clusters, of noise points and the largest cluster.

**5c.** At eps = 0.5 on `Z2`, what share of the noise points are malignant, against 37% overall?

In [ ]:
# provided
Z2 = PCA(2, random_state=SEED).fit_transform(Z)

In [ ]:
# your code here


**Question 5.** Why does DBSCAN fail in 30 dimensions here, and what kind of biological data would suit it?

*Your answer here.*

## Task 6. Linkage and the linkage trap (15 min)

**6a.** For single, complete, average and Ward linkage on `Z` (`scipy.cluster.hierarchy.linkage`), cut each tree into two
clusters (`fcluster(..., 2, criterion="maxclust")`) and report the sizes, the silhouette and the ARI with diagnosis.

**6b.** Draw the four dendrograms for the 60 tumours in `idx60` (provided) side by side.

**6c.** Which tumours does single linkage separate? Print their mean area and compare with the median.

In [ ]:
# provided: the 60 tumours used on the website and in the lecture figure
idx60 = np.sort(np.random.default_rng(0).choice(len(Z), 60, replace=False))

In [ ]:
# your code here


**Question 6.** Single linkage has the highest silhouette. Why is it the least useful answer, and what habit protects you?

*Your answer here.*

## Task 7. Manufacture a batch effect (25 min): the heart of the lab

Batch effects are technical shifts shared by samples processed together. We **simulate** one on the real tumours: half
the tumours are put in "batch B" and every raw feature of batch B is shifted by `shift` × that feature's standard
deviation, in a direction fixed per feature. Two designs:
* **random**: batch B is a coin toss, independent of diagnosis;
* **confounded**: about 80% of malignant and 20% of benign tumours are in batch B (as when cases and controls are
  collected at different times).

**7a.** Write `run(assign, shift, correct=False)`: shift the raw data, optionally centre each batch on its own mean,
standardise, run k-means (k = 2, `n_init=10, random_state=SEED`) and return the silhouette, the ARI with diagnosis and the
ARI with batch.

**7b.** For the random design, sweep `shift` over `SHIFTS` and plot the three numbers. Where do the clusters switch?

**7c.** At shift 1.5, compare: random; random and centred; confounded; confounded and centred.

In [ ]:
# provided: the batch assignments and the shift directions (seeded, as in the lecture)
rng7 = np.random.default_rng(7)
batch_random = rng7.integers(0, 2, len(y))
batch_conf = np.where(y == 1, rng7.random(len(y)) < 0.8, rng7.random(len(y)) < 0.2).astype(int)
SIGN = np.random.default_rng(3).choice([-1, 1], X_raw.shape[1])
SD = X_raw.std(axis=0)
SHIFTS = [0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]
print("confounded design: share of malignant in batch B", batch_conf[y == 1].mean().round(3),
      "; share of benign", batch_conf[y == 0].mean().round(3))

In [ ]:
# your code here


**Question 7.** (a) At what shift do the clusters stop following the diagnosis, and what happens to the silhouette?
(b) Why does centring each batch rescue the random design but not the confounded one? (c) What should have been done
when the samples were collected?

*Your answer here.*

## Task 8. Clusters in protein embedding space (15 min)

In Week 7, k-NN on the ESM-2 embeddings predicted signal peptides and membrane proteins well (test ROC AUC 0.94).
Do k-means clusters of the same embeddings follow those labels?

**8a.** Build three representations (provided): the 23 classical descriptors, standardised; ESM-2, standardised
(Euclidean); and ESM-2 centred and scaled to unit length (cosine).

**8b.** For k = 2 to 8, run k-means (`n_init=10, random_state=SEED`) on each and record the ARI with the signal-peptide
label, the ARI with the membrane label, and η², the share of the variance in log length explained by cluster
membership (function provided).

**8c.** For the classical descriptors at k = 3, print each cluster's size, share with a signal peptide, share membrane
and median length.

In [ ]:
# provided
sp = prot.has_signal_peptide.values; tm = prot.is_transmembrane.values
loglen = np.log(prot.length.values)
CLASSICAL = [c for c in prot.columns if c.startswith("frac_")] + ["log_length", "max_hydropathy_19", "nterm_hydropathy_25"]
E = emb.values
REPS = {"classical": StandardScaler().fit_transform(prot[CLASSICAL].values),
        "esm2": StandardScaler().fit_transform(E),
        "esm2_cosine": normalize(E - E.mean(axis=0))}

def eta2(v, lab):
    tot = ((v - v.mean()) ** 2).sum()
    return sum((lab == c).sum() * (v[lab == c].mean() - v.mean()) ** 2 for c in np.unique(lab)) / tot
print({k: v.shape for k, v in REPS.items()})

In [ ]:
# your code here


**Question 8.** k-NN on ESM-2 predicts both labels well, yet ESM-2 clusters barely follow them. Explain the apparent contradiction, and say what the clusters do follow.

*Your answer here.*

## Task 9. What can you claim? (10 min)

**9a.** Write the two sentences you would need in a paper to claim that the k = 2 WDBC clusters are biologically
meaningful subgroups. Then say honestly whether your analysis supports them.

**9b.** Using the lecture's checklist (technical variables; robustness; stability; outcome not used to cluster;
coherent biology; independent replication), tick which checks this lab performed and which it could not.

*Your answer here.*

## Optional extensions

**A. Other algorithms.** Fit `GaussianMixture(n_components=k)` on `Z2` and compare BIC across k; then run
`sklearn.cluster.HDBSCAN(min_cluster_size=15)` on `Z2`. Do they agree with k-means and DBSCAN?

**B. Cluster-wise stability (Hennig 2007).** For k = 2 and 3, compute for each cluster the mean Jaccard similarity
between it and its best-matching cluster across 50 bootstrap resamples. Are some clusters stable and others not?

**C. PCA before clustering proteins.** Reduce standardised ESM-2 to 10 or 50 principal components before k-means. Do
the ARIs with the labels change?

In [ ]:
# your code here (optional)
